In [ ]:
# ============================================================
# CELL 1--NOTEBOOK 5 — SHAP PRUNE
# ============================================================

import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import joblib

from pathlib import Path

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Project directories
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
FIELD_DIR = DATA_DIR / "field"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURE_DIR = RESULTS_DIR / "figures"
TABLE_DIR = RESULTS_DIR / "tables"
REPORT_DIR = RESULTS_DIR / "reports"
# Create output directories
for folder in [FIGURE_DIR, TABLE_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


In [2]:
#Cell 3 — Load the locked baseline data
# ============================================================
# LOAD BINARY MODELLING DATA
# ============================================================

train_binary_model = pd.read_csv(
    PROCESSED_DIR / "train_binary_model.csv"
)

test_binary_model = pd.read_csv(
    PROCESSED_DIR / "test_binary_model.csv"
)

TARGET = "RISK_BINARY"

X_train = train_binary_model.drop(
    columns=[TARGET]
)

y_train = train_binary_model[TARGET].copy()

X_test = test_binary_model.drop(
    columns=[TARGET]
)

y_test = test_binary_model[TARGET].copy()

print("=" * 70)
print("SHAP DATA LOADED")
print("=" * 70)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

SHAP DATA LOADED
X_train: (468, 61)
X_test:  (117, 61)
y_train: (468,)
y_test:  (117,)


In [3]:
#Cell 4 — Load the saved baseline model
# ============================================================
# LOAD SAVED BASELINE XGBOOST
# ============================================================

baseline_model_path = (
    MODELS_DIR / "baseline_xgboost_binary.pkl"
)

assert baseline_model_path.exists()

baseline_xgb = joblib.load(
    baseline_model_path
)

print("=" * 70)
print("BASELINE MODEL LOADED")
print("=" * 70)

print(f"Model: {baseline_model_path}")

print("\n✓ Saved baseline XGBoost successfully loaded")

BASELINE MODEL LOADED
Model: ..\models\baseline_xgboost_binary.pkl

✓ Saved baseline XGBoost successfully loaded


In [4]:
#Cell 5 — Critical provenance validation
# ============================================================
# MODEL / FEATURE ALIGNMENT VALIDATION
# ============================================================

print("=" * 70)
print("MODEL / FEATURE ALIGNMENT VALIDATION")
print("=" * 70)

assert X_train.shape == (468, 61)
assert X_test.shape == (117, 61)

assert list(X_train.columns) == list(X_test.columns)

model_features = list(
    baseline_xgb.get_booster().feature_names
)

data_features = list(X_train.columns)

assert model_features == data_features

print(f"Training observations: {len(X_train)}")
print(f"Predictors:            {len(data_features)}")

print("\n✓ Model contains exactly the 61 modelling predictors")
print("✓ Model feature order matches training data")
print("✓ SHAP analysis will use training data only")

MODEL / FEATURE ALIGNMENT VALIDATION
Training observations: 468
Predictors:            61

✓ Model contains exactly the 61 modelling predictors
✓ Model feature order matches training data
✓ SHAP analysis will use training data only


In [5]:
# ============================================================
# CELL 6 — CREATE SHAP TREE EXPLAINER
# ============================================================

print("=" * 70)
print("CREATING SHAP TREE EXPLAINER")
print("=" * 70)

explainer = shap.TreeExplainer(
    baseline_xgb
)

print("✓ SHAP TreeExplainer created")
print("✓ Explainer based on the saved baseline XGBoost model")

CREATING SHAP TREE EXPLAINER
✓ SHAP TreeExplainer created
✓ Explainer based on the saved baseline XGBoost model


In [6]:
# ============================================================
# CELL 7 — CALCULATE SHAP VALUES
# ============================================================

print("=" * 70)
print("CALCULATING SHAP VALUES")
print("=" * 70)

shap_values = explainer.shap_values(
    X_train
)

print(f"SHAP array shape: {np.asarray(shap_values).shape}")
print(f"Training data shape: {X_train.shape}")

assert np.asarray(shap_values).shape == X_train.shape

print("\n✓ SHAP values calculated")
print("✓ 468 observations × 61 predictors")
print("✓ SHAP dimensions match training data")

CALCULATING SHAP VALUES
SHAP array shape: (468, 61)
Training data shape: (468, 61)

✓ SHAP values calculated
✓ 468 observations × 61 predictors
✓ SHAP dimensions match training data


In [7]:
#Cell 8 — Calculate mean absolute SHAP importance
# ============================================================
# CELL 8 — SHAP FEATURE IMPORTANCE
# ============================================================

shap_array = np.asarray(shap_values)

mean_abs_shap = np.mean(
    np.abs(shap_array),
    axis=0
)

assert len(mean_abs_shap) == 61

shap_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Mean_Abs_SHAP": mean_abs_shap
})

# Rank features from most to least important
shap_importance = shap_importance.sort_values(
    by="Mean_Abs_SHAP",
    ascending=False
).reset_index(drop=True)

# Add rank
shap_importance["SHAP_Rank"] = (
    np.arange(len(shap_importance)) + 1
)

print("=" * 70)
print("SHAP FEATURE IMPORTANCE")
print("=" * 70)

print(
    shap_importance[
        ["SHAP_Rank", "Feature", "Mean_Abs_SHAP"]
    ].to_string(index=False)
)

print("\n✓ 61 predictors ranked by mean |SHAP|")

SHAP FEATURE IMPORTANCE
 SHAP_Rank                       Feature  Mean_Abs_SHAP
         1                        GPA_S1       1.797494
         2       Persists when difficult       0.389422
         3             Takes rest breaks       0.328431
         4                        CA_AVG       0.302682
         5      Manages time effectively       0.289448
         6                      ATT_RATE       0.269190
         7  Confident in clinical skills       0.251732
         8     Anxious about assessments       0.231400
         9                      EXAM_AVG       0.230998
        10      Reviews notes within 24h       0.225499
        11                      CLIN_AVG       0.222235
        12         Participates in clubs       0.193795
        13            Sense of belonging       0.177523
        14    Confident to complete prog       0.177002
        15                       LAB_AVG       0.174530
        16        Lecturers approachable       0.172374
        17   Concentrati

In [8]:
# ============================================================
# CELL 9 — CUMULATIVE SHAP CONTRIBUTION
# ============================================================

total_shap = (
    shap_importance["Mean_Abs_SHAP"].sum()
)

shap_importance["SHAP_Contribution"] = (
    shap_importance["Mean_Abs_SHAP"] /
    total_shap
)

shap_importance["Cumulative_SHAP"] = (
    shap_importance["SHAP_Contribution"].cumsum()
)

print("=" * 70)
print("CUMULATIVE SHAP CONTRIBUTION")
print("=" * 70)

print(
    shap_importance[
        [
            "SHAP_Rank",
            "Feature",
            "Mean_Abs_SHAP",
            "SHAP_Contribution",
            "Cumulative_SHAP"
        ]
    ].to_string(index=False)
)

CUMULATIVE SHAP CONTRIBUTION
 SHAP_Rank                       Feature  Mean_Abs_SHAP  SHAP_Contribution  Cumulative_SHAP
         1                        GPA_S1       1.797494           0.189168         0.189168
         2       Persists when difficult       0.389422           0.040983         0.230151
         3             Takes rest breaks       0.328431           0.034564         0.264715
         4                        CA_AVG       0.302682           0.031854         0.296569
         5      Manages time effectively       0.289448           0.030461         0.327030
         6                      ATT_RATE       0.269190           0.028330         0.355360
         7  Confident in clinical skills       0.251732           0.026492         0.381852
         8     Anxious about assessments       0.231400           0.024353         0.406205
         9                      EXAM_AVG       0.230998           0.024310         0.430515
        10      Reviews notes within 24h       0.22

In [9]:
# ============================================================
# CELL 10 — 95% SHAP FEATURE SELECTION
# ============================================================

SHAP_THRESHOLD = 0.95

# Smallest number of features reaching or exceeding 95%
selected_count = (
    shap_importance["Cumulative_SHAP"]
    .ge(SHAP_THRESHOLD)
    .idxmax()
    + 1
)

selected_features = (
    shap_importance
    .iloc[:selected_count]
    ["Feature"]
    .tolist()
)

# Mark selected features
shap_importance["Selected"] = (
    shap_importance["SHAP_Rank"] <= selected_count
)

achieved_cumulative = (
    shap_importance
    .iloc[selected_count - 1]
    ["Cumulative_SHAP"]
)

print("=" * 70)
print("SHAP FEATURE SELECTION")
print("=" * 70)

print(f"SHAP threshold:        {SHAP_THRESHOLD:.2%}")
print(f"Selected features:     {selected_count}")
print(f"Achieved cumulative:   {achieved_cumulative:.4f}")
print(
    f"Features removed:      "
    f"{61 - selected_count}"
)

print("\nSelected features:")

for i, feature in enumerate(
    selected_features,
    start=1
):
    print(f"{i:02d}. {feature}")

print("\n✓ SHAP feature-selection rule applied")

SHAP FEATURE SELECTION
SHAP threshold:        95.00%
Selected features:     49
Achieved cumulative:   0.9527
Features removed:      12

Selected features:
01. GPA_S1
02. Persists when difficult
03. Takes rest breaks
04. CA_AVG
05. Manages time effectively
06. ATT_RATE
07. Confident in clinical skills
08. Anxious about assessments
09. EXAM_AVG
10. Reviews notes within 24h
11. CLIN_AVG
12. Participates in clubs
13. Sense of belonging
14. Confident to complete prog
15. LAB_AVG
16. Lecturers approachable
17. Concentration in self-study
18. Schedule conflicts
19. Hopeless/unmotivated
20. Balanced diet
21. Sleep difficulty
22. Adequate advisory support
23. Sleep_hrs
24. Programme prepares for career
25. Considered break
26. Adequate supervision
27. Sleep affects concentration
28. Rotations affect performance
29. Regular exercise
30. Uses library regularly
31. Burnt out
32. Emotionally supported
33. Early intervention provided
34. Participates in class
35. Self_risk_percep
36. Understands con

In [10]:
# ============================================================
# CELL 11 — SHAP SELECTION VALIDATION
# ============================================================

print("=" * 70)
print("SHAP SELECTION VALIDATION")
print("=" * 70)

assert len(selected_features) == selected_count

assert (
    shap_importance["Selected"].sum()
    == selected_count
)

assert achieved_cumulative >= SHAP_THRESHOLD

# The previous feature, if one exists, must be below 95%
if selected_count > 1:
    previous_cumulative = (
        shap_importance
        .iloc[selected_count - 2]
        ["Cumulative_SHAP"]
    )

    assert previous_cumulative < SHAP_THRESHOLD

print(f"✓ Total predictors:       61")
print(f"✓ Selected predictors:    {selected_count}")
print(f"✓ Removed predictors:     {61 - selected_count}")
print(f"✓ Cumulative SHAP:        {achieved_cumulative:.4f}")
print("✓ 95% threshold validated")
print("✓ Minimal feature set validated")

print("\n✓ SHAP FEATURE SELECTION VALIDATED")

SHAP SELECTION VALIDATION
✓ Total predictors:       61
✓ Selected predictors:    49
✓ Removed predictors:     12
✓ Cumulative SHAP:        0.9527
✓ 95% threshold validated
✓ Minimal feature set validated

✓ SHAP FEATURE SELECTION VALIDATED


In [11]:
# ============================================================
# CELL 12 — SAVE SHAP PROVENANCE TABLE
# ============================================================

shap_provenance_path = (
    RESULTS_DIR / "shap_feature_provenance_binary.csv"
)

shap_importance.to_csv(
    shap_provenance_path,
    index=False
)

print("=" * 70)
print("SHAP PROVENANCE SAVED")
print("=" * 70)

print(f"File: {shap_provenance_path}")

print("\nColumns saved:")
for col in shap_importance.columns:
    print(f"  ✓ {col}")

assert shap_provenance_path.exists()

print("\n✓ Complete 61-feature SHAP provenance saved")

SHAP PROVENANCE SAVED
File: ..\results\shap_feature_provenance_binary.csv

Columns saved:
  ✓ Feature
  ✓ Mean_Abs_SHAP
  ✓ SHAP_Rank
  ✓ SHAP_Contribution
  ✓ Cumulative_SHAP
  ✓ Selected

✓ Complete 61-feature SHAP provenance saved


In [12]:
# ============================================================
# CELL 13 — SAVE SELECTED SHAP FEATURES
# ============================================================

selected_features_df = pd.DataFrame({
    "SHAP_Rank": range(1, selected_count + 1),
    "Feature": selected_features
})

selected_features_path = (
    RESULTS_DIR / "shap_selected_features_binary.csv"
)

selected_features_df.to_csv(
    selected_features_path,
    index=False
)

print("=" * 70)
print("SELECTED SHAP FEATURES SAVED")
print("=" * 70)

print(f"Selected features: {selected_count}")
print(f"File: {selected_features_path}")

assert selected_features_path.exists()

print("\n✓ 49 selected features saved")

SELECTED SHAP FEATURES SAVED
Selected features: 49
File: ..\results\shap_selected_features_binary.csv

✓ 49 selected features saved


In [13]:
# ============================================================
# CELL 14 — CREATE SHAP-PRUNED DATASETS
# ============================================================

X_train_shap = X_train[selected_features].copy()
X_test_shap = X_test[selected_features].copy()

print("=" * 70)
print("SHAP-PRUNED DATASETS")
print("=" * 70)

print(f"Original training predictors: {X_train.shape[1]}")
print(f"Selected training predictors: {X_train_shap.shape[1]}")

print(f"\nOriginal testing predictors:  {X_test.shape[1]}")
print(f"Selected testing predictors: {X_test_shap.shape[1]}")

assert X_train_shap.shape == (468, 49)
assert X_test_shap.shape == (117, 49)

print("\n✓ SHAP-pruned training dataset: 468 × 49")
print("✓ SHAP-pruned testing dataset: 117 × 49")

SHAP-PRUNED DATASETS
Original training predictors: 61
Selected training predictors: 49

Original testing predictors:  61
Selected testing predictors: 49

✓ SHAP-pruned training dataset: 468 × 49
✓ SHAP-pruned testing dataset: 117 × 49


In [14]:
# ============================================================
# CELL 15 — SHAP-PRUNED DATA VALIDATION
#Verify no information was lost incorrectly
# ============================================================

print("=" * 70)
print("SHAP-PRUNED DATA VALIDATION")
print("=" * 70)

# No missing values
assert X_train_shap.isnull().sum().sum() == 0
assert X_test_shap.isnull().sum().sum() == 0

# Numeric predictors
assert X_train_shap.select_dtypes(
    exclude=np.number
).shape[1] == 0

assert X_test_shap.select_dtypes(
    exclude=np.number
).shape[1] == 0

# Train/test alignment
assert list(
    X_train_shap.columns
) == list(
    X_test_shap.columns
)

# All selected features must originate from original predictors
assert set(selected_features).issubset(
    set(X_train.columns)
)

# No duplicate selected features
assert len(selected_features) == len(
    set(selected_features)
)

print("✓ No missing values")
print("✓ All selected predictors numeric")
print("✓ Train/test feature alignment confirmed")
print("✓ All selected predictors originate from the 61-feature space")
print("✓ No duplicate selected predictors")

print("\n✓ SHAP-PRUNED DATA VALIDATION PASSED")

SHAP-PRUNED DATA VALIDATION
✓ No missing values
✓ All selected predictors numeric
✓ Train/test feature alignment confirmed
✓ All selected predictors originate from the 61-feature space
✓ No duplicate selected predictors

✓ SHAP-PRUNED DATA VALIDATION PASSED


In [15]:
#Cell 16 — Final SHAP provenance validation
# ============================================================
# CELL 16 — NOTEBOOK 5 FINAL VALIDATION
# ============================================================

print("=" * 70)
print("NOTEBOOK 5 — FINAL SHAP VALIDATION")
print("=" * 70)

assert len(X_train.columns) == 61
assert len(selected_features) == 49
assert 61 - len(selected_features) == 12

assert achieved_cumulative >= 0.95

assert X_train_shap.shape == (468, 49)
assert X_test_shap.shape == (117, 49)

assert shap_importance.shape[0] == 61

assert shap_importance["SHAP_Rank"].is_unique

assert (
    shap_importance["Selected"].sum()
    == 49
)

print("✓ Original predictors:        61")
print("✓ Selected predictors:        49")
print("✓ Removed predictors:         12")
print("✓ Cumulative SHAP:             0.9527")
print("✓ Training SHAP data:          468 × 61")
print("✓ SHAP-pruned training data:   468 × 49")
print("✓ SHAP-pruned testing data:    117 × 49")
print("✓ Complete SHAP provenance:    saved")
print("✓ 95% selection rule:          validated")

print("\n" + "=" * 70)
print("✓ NOTEBOOK 5 SHAP VALIDATION PASSED")
print("=" * 70)

NOTEBOOK 5 — FINAL SHAP VALIDATION
✓ Original predictors:        61
✓ Selected predictors:        49
✓ Removed predictors:         12
✓ Cumulative SHAP:             0.9527
✓ Training SHAP data:          468 × 61
✓ SHAP-pruned training data:   468 × 49
✓ SHAP-pruned testing data:    117 × 49
✓ Complete SHAP provenance:    saved
✓ 95% selection rule:          validated

✓ NOTEBOOK 5 SHAP VALIDATION PASSED


In [16]:
# ============================================================
# CELL 17 — SAVE SHAP-PRUNED BINARY DATASETS
# ============================================================

print("=" * 70)
print("SAVING SHAP-PRUNED BINARY DATASETS")
print("=" * 70)

# Add target back to the SHAP-pruned predictor matrices
train_binary_shap = X_train_shap.copy()
train_binary_shap[TARGET] = y_train.values

test_binary_shap = X_test_shap.copy()
test_binary_shap[TARGET] = y_test.values

# Save datasets
train_shap_path = (
    PROCESSED_DIR / "train_binary_shap.csv"
)

test_shap_path = (
    PROCESSED_DIR / "test_binary_shap.csv"
)

train_binary_shap.to_csv(
    train_shap_path,
    index=False
)

test_binary_shap.to_csv(
    test_shap_path,
    index=False
)

print(f"Training file: {train_shap_path}")
print(f"Testing file:  {test_shap_path}")

print(f"\nTraining shape: {train_binary_shap.shape}")
print(f"Testing shape:  {test_binary_shap.shape}")

assert train_binary_shap.shape == (468, 50)
assert test_binary_shap.shape == (117, 50)

assert list(
    train_binary_shap.drop(columns=[TARGET]).columns
) == selected_features

assert list(
    test_binary_shap.drop(columns=[TARGET]).columns
) == selected_features

assert train_shap_path.exists()
assert test_shap_path.exists()

print("\n✓ SHAP-pruned training dataset saved: 468 × 50")
print("✓ SHAP-pruned testing dataset saved: 117 × 50")
print("✓ Each contains 49 predictors + RISK_BINARY")

SAVING SHAP-PRUNED BINARY DATASETS
Training file: ..\data\processed\train_binary_shap.csv
Testing file:  ..\data\processed\test_binary_shap.csv

Training shape: (468, 50)
Testing shape:  (117, 50)

✓ SHAP-pruned training dataset saved: 468 × 50
✓ SHAP-pruned testing dataset saved: 117 × 50
✓ Each contains 49 predictors + RISK_BINARY


In [17]:
# ============================================================
# CELL 18 — FINAL FILE VALIDATION
# ============================================================

print("=" * 70)
print("NOTEBOOK 5 — FINAL FILE VALIDATION")
print("=" * 70)

train_check = pd.read_csv(train_shap_path)
test_check = pd.read_csv(test_shap_path)

assert train_check.shape == (468, 50)
assert test_check.shape == (117, 50)

assert TARGET in train_check.columns
assert TARGET in test_check.columns

assert list(
    train_check.drop(columns=[TARGET]).columns
) == selected_features

assert list(
    test_check.drop(columns=[TARGET]).columns
) == selected_features

assert train_check.isnull().sum().sum() == 0
assert test_check.isnull().sum().sum() == 0

print("✓ train_binary_shap.csv: 468 × 50")
print("✓ test_binary_shap.csv:  117 × 50")
print("✓ 49 predictors + RISK_BINARY")
print("✓ Feature order verified")
print("✓ No missing values")
print("✓ Files successfully reloaded")

print("\n" + "=" * 70)
print("✓ NOTEBOOK 5 FILE SAVE VALIDATION PASSED")
print("=" * 70)

NOTEBOOK 5 — FINAL FILE VALIDATION
✓ train_binary_shap.csv: 468 × 50
✓ test_binary_shap.csv:  117 × 50
✓ 49 predictors + RISK_BINARY
✓ Feature order verified
✓ No missing values
✓ Files successfully reloaded

✓ NOTEBOOK 5 FILE SAVE VALIDATION PASSED
